<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Konvergencia%20Learning%20Rates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Konvergencia és Learning Rates

Ebben a notebookban a **konvergencia és learning rate kapcsolatát** vizsgáljuk.

## Tartalomjegyzék

1. Konvergencia feltételei
2. Learning rate és konvergencia sebesség
3. Konvergencia diagnosztika
4. Learning rate vs batch size
5. Gyakorlati tippek

## 1. Konvergencia feltételei

### Konvex optimalizáció

SGD konvergenciája konvex $L$-smooth függvényre:

$$E[L(\theta_T) - L(\theta^*)] \leq \frac{||\theta_0 - \theta^*||^2}{2\eta T} + \frac{\eta L \sigma^2}{2}$$

ahol:
- $L$: Lipschitz konstans (simított)
- $\sigma^2$: gradiens variancia
- $\eta$: learning rate

### Optimális learning rate

$$\eta^* = \frac{||\theta_0 - \theta^*||}{\sigma \sqrt{LT}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

# Konvex probléma: lineáris regresszió
def generate_linear_data(n_samples=1000, n_features=10, noise=0.1):
    X = np.random.randn(n_samples, n_features)
    true_w = np.random.randn(n_features)
    y = X @ true_w + noise * np.random.randn(n_samples)
    return X, y, true_w

X, y, true_w = generate_linear_data()

# SGD konvergencia különböző lr-ekkel
def sgd_linear_regression(X, y, lr, epochs, batch_size=32):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    losses = []

    for epoch in range(epochs):
        # Shuffle
        idx = np.random.permutation(n_samples)
        X_shuffled = X[idx]
        y_shuffled = y[idx]

        for i in range(0, n_samples, batch_size):
            X_batch = X_shuffled[i:i+batch_size]
            y_batch = y_shuffled[i:i+batch_size]

            # Gradiens
            pred = X_batch @ w
            grad = -2 * X_batch.T @ (y_batch - pred) / len(y_batch)

            # Update
            w = w - lr * grad

        # Epoch loss
        loss = np.mean((y - X @ w) ** 2)
        losses.append(loss)

    return w, losses

# Különböző learning rate-ek
lrs = [0.0001, 0.001, 0.01, 0.1, 0.5]
epochs = 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr in lrs:
    w, losses = sgd_linear_regression(X, y, lr, epochs)
    axes[0].plot(losses, label=f'lr={lr}')

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Konvergencia különböző lr-ekkel')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Végső loss vs lr
final_losses = []
for lr in np.logspace(-5, 0, 50):
    try:
        w, losses = sgd_linear_regression(X, y, lr, 50)
        if np.isnan(losses[-1]) or losses[-1] > 1e10:
            final_losses.append(np.nan)
        else:
            final_losses.append(losses[-1])
    except:
        final_losses.append(np.nan)

axes[1].plot(np.logspace(-5, 0, 50), final_losses, 'b-', linewidth=2)
axes[1].set_xlabel('Learning Rate')
axes[1].set_ylabel('Final MSE Loss')
axes[1].set_title('Végső loss vs Learning Rate')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Learning rate és konvergencia sebesség

### Trade-off

| Lr | Konvergencia | Stabilitás | Végső minőség |
|-----|--------------|------------|---------------|
| Kicsi | Lassú | Stabil | Jó (lokális min) |
| Nagy | Gyors | Instabil | Rossz |
| Optimális | Gyors | Stabil | Legjobb |

In [ ]:
# Konvergencia sebesség analízis

def convergence_time(losses, threshold=0.01):
    """Hány epoch kell a threshold eléréséhez."""
    for i, loss in enumerate(losses):
        if loss < threshold:
            return i
    return len(losses)  # Nem érte el

lrs = np.logspace(-4, -1, 20)
conv_times = []
final_losses = []

for lr in lrs:
    w, losses = sgd_linear_regression(X, y, lr, 200)
    conv_times.append(convergence_time(losses, threshold=0.02))
    final_losses.append(min(losses) if not np.isnan(losses[-1]) else np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Konvergencia idő
axes[0].plot(lrs, conv_times, 'bo-', linewidth=2, markersize=6)
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Epochok a konvergenciáig')
axes[0].set_title('Konvergencia sebesség')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)

# Végső minőség
axes[1].plot(lrs, final_losses, 'ro-', linewidth=2, markersize=6)
axes[1].set_xlabel('Learning Rate')
axes[1].set_ylabel('Legjobb MSE Loss')
axes[1].set_title('Végső minőség')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Konvergencia diagnosztika

### Jelek és jelentések

| Viselkedés | Diagnózis | Megoldás |
|------------|-----------|----------|
| Oszcilláció | Lr túl nagy | Csökkentés |
| Lapos görbe | Lr túl kicsi | Növelés |
| NaN | Explodáló gradiens | Gradient clipping |
| Platók | Lokális minimum | Momentum, lr bump |

In [ ]:
# Diagnosztikai példák

# Nem-konvex probléma: neurális háló
X_nn, y_nn = make_regression(n_samples=1000, n_features=20, noise=10, random_state=42)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_nn = scaler_X.fit_transform(X_nn)
y_nn = scaler_y.fit_transform(y_nn.reshape(-1, 1)).flatten()

X_nn_t = torch.FloatTensor(X_nn)
y_nn_t = torch.FloatTensor(y_nn).unsqueeze(1)

def train_nn(lr, epochs=100):
    torch.manual_seed(42)
    model = nn.Sequential(
        nn.Linear(20, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    )
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    losses = []
    grad_norms = []

    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_nn_t)
        loss = criterion(outputs, y_nn_t)
        loss.backward()

        # Gradiens norma
        total_norm = 0
        for p in model.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        grad_norms.append(np.sqrt(total_norm))

        optimizer.step()
        losses.append(loss.item())

    return losses, grad_norms

# Különböző lr-ek
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

test_lrs = [0.0001, 0.01, 1.0]
labels = ['Túl kicsi (0.0001)', 'Jó (0.01)', 'Túl nagy (1.0)']

for col, (lr, label) in enumerate(zip(test_lrs, labels)):
    losses, grad_norms = train_nn(lr, epochs=100)

    # Loss
    axes[0, col].plot(losses, 'b-', linewidth=2)
    axes[0, col].set_xlabel('Epoch')
    axes[0, col].set_ylabel('Loss')
    axes[0, col].set_title(f'{label}\nVégső loss: {losses[-1]:.4f}')
    axes[0, col].grid(True, alpha=0.3)

    # Gradiens norma
    axes[1, col].plot(grad_norms, 'r-', linewidth=2)
    axes[1, col].set_xlabel('Epoch')
    axes[1, col].set_ylabel('Gradient Norm')
    axes[1, col].set_title('Gradiens norma')
    axes[1, col].grid(True, alpha=0.3)

plt.suptitle('Konvergencia diagnosztika', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Learning rate vs batch size

### Linear scaling rule

$$\eta_{new} = \eta_{base} \cdot \frac{B_{new}}{B_{base}}$$

Ha batch size-t növeled, lr-t is arányosan növeld.

### Square root scaling

$$\eta_{new} = \eta_{base} \cdot \sqrt{\frac{B_{new}}{B_{base}}}$$

Konzervatívabb megközelítés.

In [ ]:
# Batch size és lr kapcsolata

def train_with_batch_size(batch_size, lr, epochs=50):
    torch.manual_seed(42)
    model = nn.Sequential(
        nn.Linear(20, 64),
        nn.ReLU(),
        nn.Linear(64, 1)
    )
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    losses = []
    n_samples = len(X_nn_t)

    for epoch in range(epochs):
        # Mini-batch training
        epoch_loss = 0
        n_batches = 0

        idx = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            batch_idx = idx[i:i+batch_size]
            X_batch = X_nn_t[batch_idx]
            y_batch = y_nn_t[batch_idx]

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        losses.append(epoch_loss / n_batches)

    return losses

# Összehasonlítás
base_batch = 32
base_lr = 0.01

batch_sizes = [32, 64, 128, 256]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Fix lr
for bs in batch_sizes:
    losses = train_with_batch_size(bs, base_lr)
    axes[0].plot(losses, label=f'BS={bs}')
axes[0].set_title('Fix lr=0.01')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Linear scaling
for bs in batch_sizes:
    scaled_lr = base_lr * bs / base_batch
    losses = train_with_batch_size(bs, scaled_lr)
    axes[1].plot(losses, label=f'BS={bs}, lr={scaled_lr:.4f}')
axes[1].set_title('Linear scaling')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Square root scaling
for bs in batch_sizes:
    scaled_lr = base_lr * np.sqrt(bs / base_batch)
    losses = train_with_batch_size(bs, scaled_lr)
    axes[2].plot(losses, label=f'BS={bs}, lr={scaled_lr:.4f}')
axes[2].set_title('Square root scaling')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Batch size és Learning Rate skálázás', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Gyakorlati tippek

### Learning rate kiválasztás

| Lépés | Teendő |
|-------|--------|
| 1 | LR finder futtatása |
| 2 | Warm-up használata |
| 3 | Scheduler alkalmazása |
| 4 | Monitoring |

### Tipikus értékek

| Optimalizáló | Alapértelmezett lr |
|--------------|-------------------|
| SGD | 0.01 - 0.1 |
| Adam | 0.001 |
| AdamW | 0.001 |

In [ ]:
# Gyakorlati példa: LR finder + OneCycleLR

from torch.optim.lr_scheduler import OneCycleLR

# Teljes tanítási pipeline
def train_with_schedule(max_lr, epochs=100):
    torch.manual_seed(42)
    model = nn.Sequential(
        nn.Linear(20, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1)
    )

    optimizer = optim.Adam(model.parameters(), lr=max_lr/25)  # OneCycle div factor
    scheduler = OneCycleLR(optimizer, max_lr=max_lr, epochs=epochs, steps_per_epoch=1)
    criterion = nn.MSELoss()

    losses = []
    lrs = []

    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_nn_t)
        loss = criterion(outputs, y_nn_t)
        loss.backward()
        optimizer.step()
        scheduler.step()

        losses.append(loss.item())
        lrs.append(optimizer.param_groups[0]['lr'])

    return losses, lrs

# Fix lr vs OneCycleLR
losses_fixed, _ = train_nn(0.01, epochs=100)
losses_cycle, lrs_cycle = train_with_schedule(0.01, epochs=100)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Loss összehasonlítás
axes[0].plot(losses_fixed, 'b-', linewidth=2, label='Fix lr=0.01')
axes[0].plot(losses_cycle, 'r-', linewidth=2, label='OneCycleLR')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss összehasonlítás')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# LR schedule
axes[1].plot(lrs_cycle, 'g-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('OneCycleLR schedule')
axes[1].grid(True, alpha=0.3)

# Végső eredmény
ax2 = axes[2]
methods = ['Fix LR', 'OneCycleLR']
final = [losses_fixed[-1], losses_cycle[-1]]
colors = ['blue', 'red']
ax2.bar(methods, final, color=colors)
ax2.set_ylabel('Végső Loss')
ax2.set_title('Végső loss összehasonlítás')

for i, v in enumerate(final):
    ax2.text(i, v + 0.001, f'{v:.4f}', ha='center')

plt.tight_layout()
plt.show()

print(f"Fix lr végső loss: {losses_fixed[-1]:.6f}")
print(f"OneCycleLR végső loss: {losses_cycle[-1]:.6f}")
print(f"Javulás: {(losses_fixed[-1] - losses_cycle[-1]) / losses_fixed[-1] * 100:.1f}%")

## Összefoglalás

### Konvergencia és LR:

| Kapcsolat | Leírás |
|-----------|--------|
| Nagy lr | Gyors de instabil |
| Kicsi lr | Lassú de stabil |
| Scheduler | Mindkettő előnye |

### Batch size skálázás:

| Szabály | Képlet |
|---------|--------|
| Linear | $\eta \propto B$ |
| Square root | $\eta \propto \sqrt{B}$ |

### Praktikus tanácsok:

1. LR finder használata
2. Warm-up alkalmazása
3. Scheduler (OneCycleLR)
4. Gradiens norma monitorozás